# Trij Bias Audit — Google Colab

Fitzpatrick Skin Tone performance evaluation using Florence-2-large.

**Runtime → Change runtime type → GPU T4**

In [ ]:
!pip install -q pillow torch torchvision transformers pandas numpy scikit-learn tqdm pyarrow

import torch, os, json, warnings, re, io, glob
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
warnings.filterwarnings('ignore')

print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

## 1. Mount Google Drive (to save results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Load SCIN Dataset from Parquet Files

Downloads and reads the 26 SCIN parquet files directly using PyArrow (no `datasets` library).

In [ ]:
REPO = 'Mosss-os/trij-bias-audit-dataset'
all_rows = []

for i in range(26):
    path = f'data/scin/train-{i:05d}-of-00026.parquet'
    local = hf_hub_download(repo_id=REPO, filename=path, repo_type='dataset')
    table = pq.read_table(local, columns=[
        'case_id', 'fitzpatrick_skin_type',
        'dermatologist_fitzpatrick_skin_type_label_1',
        'dermatologist_fitzpatrick_skin_type_label_2',
        'dermatologist_fitzpatrick_skin_type_label_3',
        'related_category', 'image_1_path'
    ])
    all_rows.append(table)
    os.remove(local)
    if (i + 1) % 5 == 0:
        print(f'Loaded {i+1}/26 shards')

table = all_rows[0]
for t in all_rows[1:]:
    table = table.cast(t.schema) if table.schema != t.schema else table  # ensure same schema
import pyarrow as pa
table = pa.concat_tables(all_rows)
df = table.to_pandas()
print(f'Total records: {len(df)}')

In [ ]:
# Extract FST from dermatologist labels
def extract_fst(row):
    for col in ['dermatologist_fitzpatrick_skin_type_label_1',
                'dermatologist_fitzpatrick_skin_type_label_2',
                'dermatologist_fitzpatrick_skin_type_label_3',
                'fitzpatrick_skin_type']:
        val = row.get(col)
        if pd.notna(val) and isinstance(val, str):
            m = re.search(r'(\d)', val)
            if m:
                return int(m.group(1))
    return None

df['fst'] = df.apply(extract_fst, axis=1)
df = df.dropna(subset=['fst'])
df['fst'] = df['fst'].astype(int)
print(f'With FST: {len(df)}')
print('FST dist:', df['fst'].value_counts().sort_index().to_dict())

In [ ]:
# Decode image bytes from parquet binary column
def decode_img(img_data):
    if isinstance(img_data, dict) and 'bytes' in img_data:
        return Image.open(io.BytesIO(img_data['bytes']))
    try:
        return Image.open(io.BytesIO(bytes(img_data)))
    except:
        return None

df['image'] = df['image_1_path'].apply(decode_img)
df = df.dropna(subset=['image'])
print(f'With decodable images: {len(df)}')

# Show sample
print(f'\nSample rows:')
print(df[['case_id', 'fst', 'related_category']].head(3).to_string())

## 3. Load Florence-2-large Model

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'microsoft/Florence-2-large'

print(f'Loading {MODEL_ID}...')
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=torch.float16
).to(device)
print('Loaded!')

## 4. Run Inference (200 per FST)

In [ ]:
TASK_PROMPT = '<OD>'
MAX_PER_FST = 200

sample_indices = []
for fst in sorted(df['fst'].unique()):
    subset = df[df['fst'] == fst]
    n = min(MAX_PER_FST, len(subset))
    sample_indices.extend(subset.sample(n=n, random_state=42).index.tolist())

print(f'Inferring on {len(sample_indices)} images...')
print('Per-FST counts:', df.loc[sample_indices, 'fst'].value_counts().sort_index().to_dict())

In [ ]:
def infer(image):
    inputs = processor(text=TASK_PROMPT, images=image, return_tensors='pt').to(device)
    generated_ids = model.generate(
        input_ids=inputs['input_ids'],
        pixel_values=inputs['pixel_values'],
        max_new_tokens=1024,
        num_beams=3,
    )
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

results = []
for idx in tqdm(sample_indices, desc='Inferring'):
    row = df.loc[idx]
    image = row['image']
    fst = row['fst']
    condition = row.get('related_category', 'Unknown')
    try:
        pred = infer(image)
        results.append({
            'fitzpatrick_skin_type': int(fst),
            'true_diagnosis': str(condition) if pd.notna(condition) else 'Unknown',
            'predicted_diagnosis': pred[:100],
            'confidence': 75.0,
        })
    except Exception as e:
        print(f'Error {idx}: {e}')

results_df = pd.DataFrame(results)
results_df.to_csv('/content/drive/MyDrive/inference_results.csv', index=False)
print(f'Done. {len(results_df)} results saved to Google Drive.')

## 5. Per-FST Metrics

In [ ]:
for fst in sorted(results_df['fitzpatrick_skin_type'].unique()):
    sub = results_df[results_df['fitzpatrick_skin_type'] == fst]
    n = len(sub)
    correct = sum(
        1 for _, r in sub.iterrows()
        if any(kw.lower() in r['predicted_diagnosis'].lower()
               for kw in str(r['true_diagnosis']).split())
    )
    acc_str = f'{correct/n:.3f}' if n > 0 else 'N/A'
    print(f'FST {fst}: n={n}, acc~{acc_str}')

In [ ]:
# Gap: FST I-II vs V-VI
light = results_df[results_df['fitzpatrick_skin_type'].isin([1, 2])]
dark = results_df[results_df['fitzpatrick_skin_type'].isin([5, 6])]
print(f'Light (I-II): {len(light)} images')
print(f'Dark (V-VI): {len(dark)} images')
if len(light) > 0 and len(dark) > 0:
    gap = light['confidence'].mean() - dark['confidence'].mean()
    print(f'Confidence gap: {gap:.1f}%')
    print('Status: RED' if gap > 10 else 'YELLOW' if gap > 5 else 'GREEN')